## Vector Database in the RAG Pipeline

A **Vector Database** stores the **embeddings (vectors)** of document chunks. Instead of searching by exact keywords, it searches by **meaning (semantic similarity)** to find the most relevant information.

### Why Do We Need It?

Suppose your company has thousands of documents.

A user asks:

> **Can interns work remotely?**

Instead of searching every document, the vector database quickly finds the most relevant chunk.

---

### RAG Pipeline

```text
Documents
    │
    ▼
Load Documents
    │
    ▼
Split into Chunks
    │
    ▼
Generate Embeddings
    │
    ▼
Store in Vector Database
    ▲
    │
User Question
    │
    ▼
Generate Query Embedding
    │
    ▼
Similarity Search
    │
    ▼
Retrieve Top Chunks
    │
    ▼
LLM
    │
    ▼
Final Answer
```

---

### Real-Life Example

Stored document:

```
Interns may work remotely
with manager approval.
```

User asks:

> **Can interns work from home?**

The vector database understands that **"work remotely"** and **"work from home"** have similar meanings and retrieves the correct document.

The LLM then uses that context to generate an accurate answer.

---

### Popular Vector Databases

- PostgreSQL + pgvector
- Pinecone
- Qdrant
- Chroma
- FAISS
- Weaviate

---

### Summary

- Stores **document embeddings**
- Performs **semantic search**
- Retrieves the most relevant document chunks
- Provides context to the LLM
- Helps reduce hallucinations in RAG applications


In [ ]:
import os
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader,
    PyPDFLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from pathlib import Path
import hashlib
from langchain_google_genai import GoogleGenerativeAIEmbeddings
EMBEDDING_MODEL =os.getenv('EMBEDDING_MODEL')
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

C:\Users\kisho\AppData\Local\Temp\ipykernel_19236\2053770860.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
c:\Users\kisho\WorkSpace\learning\ai workflow\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

embeddings_model = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GEMINI_API_KEY,
)
embeddings_model


GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x000001CDB0A6ABA0>, model='gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [ ]:
### Read all the PDF with the directory
def process_pdfs(pdf_directory):
    """Process all PDFs in within a Directory"""

    all_docs = []
    pdf_dir = Path(pdf_directory).resolve()
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found PDF files: {len(pdf_files)}")
    for pdf_file in pdf_files:
        # Process each PDF file
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            docs = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = str(pdf_file.name)
                doc.metadata["file_type"] = str(pdf_file.suffix)
            all_docs.extend(docs)
            print(f"FIle Loaded with {len(documents)} documents {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents loaded: {len(all_docs)}")
    return all_docs


all_pdf_docs = process_pdfs("data")
# all_pdf_docs

Found PDF files: 2
Processing AI Engineer Roadmap 2026.pdf
FIle Loaded with 40 documents AI Engineer Roadmap 2026.pdf
Processing EasyDEV.pdf
FIle Loaded with 6 documents EasyDEV.pdf
Total documents loaded: 46


In [ ]:
### Text Splitting
def split_documents(documents, chunk_size=550, chunk_overlap=80):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)
    print(f" {len(documents)} documnets is Split into {len(split_docs)} chunks")
    # return split_docs
    if split_docs:
        print("\nExample chunks:")
        print(
            f"Content: {split_docs[0].page_content[:100]}..."
        )  # Print first 100 characters
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs


#         for i, doc in enumerate(split_docs[:3]):  # Print first 3 chunks
#             print(f"Chunk {i+1}:\n{doc.page_content}\n")

split_chunks = split_documents(all_pdf_docs)

 46 documnets is Split into 151 chunks

Example chunks:
Content: Transform a production-grade full stack engineer
(React/Next.js/Node.js/TypeScript/PostgreSQL/Redis/...
Metadata: {'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-06-26T03:33:32+00:00', 'source': 'C:\\Users\\kisho\\WorkSpace\\learning\\ai workflow\\langchain\\RAG\\data\\pdf\\AI Engineer Roadmap 2026.pdf', 'file_path': 'C:\\Users\\kisho\\WorkSpace\\learning\\ai workflow\\langchain\\RAG\\data\\pdf\\AI Engineer Roadmap 2026.pdf', 'total_pages': 40, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-26T03:33:32+00:00', 'trapped': '', 'modDate': "D:20260626033332+00'00'", 'creationDate': "D:20260626033332+00'00'", 'page': 0}


## Embeddings

An **embedding** is a numerical representation (vector) of text that captures its **meaning**. Instead of storing words as plain text, an embedding model converts them into numbers that a computer can compare.

This allows AI systems to search by **meaning (semantic similarity)** instead of exact keywords.

---

### Example

Text:

```
I love dogs.
```

Embedding:

```text
[0.23, -0.45, 0.78, 0.12, ...]
```

Another text:

```
I like puppies.
```

Embedding:

```text
[0.21, -0.43, 0.80, 0.10, ...]
```

Although the words are different, the embeddings are very similar because both sentences have nearly the same meaning.

---

### How Embeddings Are Used in RAG

```text
Document
     │
     ▼
Embedding Model
     │
     ▼
Embedding Vector
     │
     ▼
Vector Database
```

When a user asks a question:

```text
User Question
      │
      ▼
Generate Query Embedding
      │
      ▼
Similarity Search
      │
      ▼
Retrieve Relevant Documents
```

---

### Popular Embedding Models

- OpenAI `text-embedding-3-small`
- OpenAI `text-embedding-3-large`
- Google Gemini Embeddings
- BGE
- E5
- Nomic Embed
- Qwen Embeddings

---

### Key Takeaway

**Embeddings convert text into vectors so computers can understand and compare the meaning of text, making semantic search possible in RAG systems.**


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from time import sleep

from langchain_community import embeddings


class EmbeddingManager:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        """
        Initialize the EmbeddingManager with a specified model name.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Load Google Model.
        """
        try:
            print("Loading Google EEmbedding model...")
            self.model = embeddings_model
             # Generate one sample embedding to check everything is working
            sample = self.model.embed_query("Hello, my dog is cute")
            print(f"Model Loaded Successfully: {self.model_name}")
            print(f"Embedding Dimension: {len(sample)}")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")
            raise
    

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of text strings to embed.

        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")


        embeddings = []

        for i, text in enumerate(texts, start=1):
         print(f"Generating embedding {i}/{len(texts)}")

         embedding = self.model.embed_query(text)
         embeddings.append(embedding)

        # Wait 1 seconds before the next request
         if i < len(texts):
            print("Waiting 1 seconds...")
            sleep(0.8)

         print("Embeddings generated successfully.")

        return np.array(embeddings)

    def get_embedding_dimension(self) -> int:
        """Get the dimensionality of the embeddings."""
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager

Loading Google EEmbedding model...
Model Loaded Successfully: gemini-embedding-001
Embedding Dimension: 3072


# Vector Store

## Overview

A **Vector Store** (also called a **Vector Database**) is a storage system that stores **embeddings (vectors)** instead of plain text. It allows applications to perform **semantic search**, meaning it retrieves information based on the **meaning** of the text rather than exact keywords.

Vector stores are a core component of **Retrieval-Augmented Generation (RAG)** systems.

---

## Why Do We Use a Vector Store?

Traditional keyword search has limitations:

- Requires exact keyword matches.
- Doesn't understand context or synonyms.
- Becomes inefficient with large datasets.

A vector store solves these problems by:

- Understanding the meaning of text.
- Finding semantically similar documents.
- Retrieving relevant context quickly.
- Improving LLM responses with accurate information.

---

## How Does It Work?

The process is simple:

```text
Documents
    │
    ▼
Split into Chunks
    │
    ▼
Generate Embeddings
    │
    ▼
Store in Vector Store
    │
    ▼
User Query
    │
    ▼
Generate Query Embedding
    │
    ▼
Similarity Search
    │
    ▼
Top Relevant Chunks
    │
    ▼
LLM Generates Final Answer
```

---

## Example

Suppose your knowledge base contains:

```text
Document 1:
LangGraph is used to build AI agents.

Document 2:
PostgreSQL is a relational database.

Document 3:
RAG combines retrieval with LLMs.
```

User asks:

```text
How can I build an AI agent?
```

Instead of searching for the exact words, the vector store understands the meaning of the query and returns:

```text
✅ Document 1:
LangGraph is used to build AI agents.
```

The retrieved document is then provided to the LLM, which uses it to generate a more accurate response.

---

## Popular Vector Stores

- FAISS
- ChromaDB
- PGVector
- Qdrant
- Milvus
- Weaviate
- Pinecone

---

## Key Takeaways

- Stores **embeddings (vectors)** instead of plain text.
- Enables **semantic similarity search**.
- Retrieves the most relevant documents for a query.
- Widely used in **RAG**, AI assistants, chatbots, and recommendation systems.
- Helps LLMs generate more accurate and context-aware responses.


In [ ]:
class VectorStore:
    """Manage all vector-related operations. and Store all Documents with embeddings within ChromaDB"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./data/vectordb_store",
    ):
        """Initialize the vector store.

        Args:
            collection_name (str): The name of the collection in ChromaDB.
            persist_directory (str): The directory where the vector store will persist data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the vector store."""
        try:
            # Create DB
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ##Get or create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection for storing PDF document embeddings",
                    "version": "1.0",
                },
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Collection metadata: {self.collection.metadata}")
            print(f"Example documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """Add a document to the vector store.

        Args:
            document (dict): The document to add, with keys "content", "metadata", and "id".
            embedding (list): The embedding for the document.

        Returns:
            None
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            source = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", 0)

            doc_id = hashlib.sha256(
                f"{source}_{page}_{i}_{doc.page_content}".encode("utf-8")
            ).hexdigest()
            ids.append(doc_id)
            # Crate Metadata
            metadata = doc.metadata.copy()  # copy is recommended
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embedding_list.append(embedding.tolist())
        try:
            self.collection.add(
                documents=documents_text,
                embeddings=embedding_list,
                metadatas=metadatas,
                ids=ids,
            )
            print(f"Successfully added {len(ids)} documents to the vector store.")
            print(f"Collection now contains {self.collection.count()} documents.")

            print("\nExample Document")
            print("-" * 40)
            print(f"ID      : {ids[0]}")
            print(f"Source  : {documents[0].metadata.get('source')}")
            print(f"Page    : {documents[0].metadata.get('page')}")
            print(f"Content : {documents[0].page_content[:100]}...")
        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

Vector store initialized with collection: pdf_documents
Collection metadata: {'description': 'Collection for storing PDF document embeddings', 'version': '1.0'}
Example documents in collection: 374


In [ ]:
split_chunks

[Document(metadata={'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-06-26T03:33:32+00:00', 'source': 'C:\\Users\\kisho\\WorkSpace\\learning\\ai workflow\\langchain\\RAG\\data\\pdf\\AI Engineer Roadmap 2026.pdf', 'file_path': 'C:\\Users\\kisho\\WorkSpace\\learning\\ai workflow\\langchain\\RAG\\data\\pdf\\AI Engineer Roadmap 2026.pdf', 'total_pages': 40, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-26T03:33:32+00:00', 'trapped': '', 'modDate': "D:20260626033332+00'00'", 'creationDate': "D:20260626033332+00'00'", 'page': 0}, page_content='Transform a production-grade full stack engineer\n(React/Next.js/Node.js/TypeScript/PostgreSQL/Redis/Docker/AWS/REST/Microservices) into a\nproduction AI engineer in 6 months (≈20 hours/week).\n1. Introduction\n2. Complete Roadmap Overview\n3. 6 Month Timeline\n4. Weekly Learning Plan (24 Weeks)\n5. Python Roadmap\n6. Mathematics for Practical AI\n7. Machine Learning Foundati

In [ ]:
### Convert the text to embedding ###

texts = [doc.page_content for doc in split_chunks]
texts

['Transform a production-grade full stack engineer\n(React/Next.js/Node.js/TypeScript/PostgreSQL/Redis/Docker/AWS/REST/Microservices) into a\nproduction AI engineer in 6 months (≈20 hours/week).\n1. Introduction\n2. Complete Roadmap Overview\n3. 6 Month Timeline\n4. Weekly Learning Plan (24 Weeks)\n5. Python Roadmap\n6. Mathematics for Practical AI\n7. Machine Learning Foundations\n8. Deep Learning Foundations\n9. Transformers Deep Dive\n10. LLM Engineering\n11. Retrieval-Augmented Generation (RAG)\n12. Vector Databases\n13. AI Agents',
 '11. Retrieval-Augmented Generation (RAG)\n12. Vector Databases\n13. AI Agents\n14. Model Context Protocol (MCP)\n15. AI Infrastructure\n16. Databases for AI Systems\n17. AI Security\n18. Observability\n19. Production AI Architecture\n20. Portfolio Projects (20 Projects)\n21. Open Source Contributions\n22. Interview Preparation\n23. Career Roadmap\n24. Certifications\nAI Engineer Roadmap 2026\n[1] [2] [3]\nTable of Contents',
 '25. Books\n26. Research 

In [ ]:
##Generate the embeddings ##

embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 151 texts...
Generating embedding 1/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 2/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 3/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 4/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 5/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 6/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 7/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 8/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 9/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 10/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 11/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 12/151
Waiting 1 seconds.

In [ ]:
##Store in Database
vector_store.add_document(split_chunks, embeddings)

Adding 151 documents to vector store...
Error adding document to vector store: Collection expecting embedding with dimension of 384, got 3072


InvalidArgumentError: Collection expecting embedding with dimension of 384, got 3072

# Retrieval in RAG (Retrieval-Augmented Generation)

## Overview

**Retrieval** is the process of finding the most relevant information from a knowledge base before sending a user's query to a Large Language Model (LLM). Instead of relying only on the model's trained knowledge, retrieval provides external context, enabling the LLM to generate more accurate and up-to-date responses.

---

## What Problem Does Retrieval Solve?

LLMs have several limitations:

- Knowledge is limited to their training data.
- Cannot answer questions about private or newly added documents.
- May generate incorrect or hallucinated responses.
- Fine-tuning for every new document is expensive.

Retrieval solves these problems by fetching relevant information from an external knowledge base at query time.

---

## How Does Retrieval Work?

The retrieval process consists of the following steps:

```text
User Question
      │
      ▼
Generate Query Embedding
      │
      ▼
Search Vector Store
      │
      ▼
Retrieve Top-K Relevant Chunks
      │
      ▼
Send Chunks + Question to LLM
      │
      ▼
Generate Final Answer
```

The vector store compares the query embedding with stored document embeddings using similarity search (e.g., Cosine Similarity) and returns the most relevant document chunks.

---

## Example

Suppose your knowledge base contains:

```text
Document 1:
LangGraph is a framework for building stateful AI agents.

Document 2:
PostgreSQL is a relational database.

Document 3:
RAG combines retrieval with LLM generation.
```

User asks:

> **"What framework can I use to build AI agents?"**

The retrieval system searches the vector database and finds the most relevant chunk:

```text
LangGraph is a framework for building stateful AI agents.
```

This retrieved context is passed to the LLM, which generates a more accurate answer.

---

## Advantages

- Provides accurate and relevant context.
- Reduces hallucinations.
- Supports private and domain-specific knowledge.
- Works with continuously updated documents.
- No need to retrain the LLM when documents change.

---

## Limitations

- Retrieval quality depends on the embedding model.
- Poor chunking can reduce search accuracy.
- Incorrect or missing documents lead to poor responses.
- Searching very large datasets may require optimized vector databases.

---

## Summary

Retrieval is the core of a RAG system. It connects user queries with the most relevant information stored in a vector database, allowing the LLM to answer using real, up-to-date, and domain-specific knowledge instead of relying solely on its pre-trained memory.


In [ ]:
from pprint import pprint
class RAGRetriever:
    """Handles query based retrieval of relevant documents. from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        """Initialize the RAGRetriever.

        Args:
            vector_store: The vector store to retrieve documents from.
            embedding_manager: The embedding manager for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 10, score_threshold: float = 0.5
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on a query.

        Args:
            query (str): The query string.
            top_k (int): The number of top documents to retrieve.
            score_threshold (float): The minimum similarity score for a document to be considered relevant.

        Returns:
            List[Document]: A list of retrieved documents.
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Score threshold: {score_threshold} and Top-k: {top_k}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )
            # pprint(results)
            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                    ):
                  
                    similarity_score = 1 - distance

                    print(f"Rank {i+1}: distance={distance:.4f}, similarity={similarity_score:.10f}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents. After filtering")
            else:
                print("No documents retrieved.")
            return retrieved_docs

        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            return []


rag_retriver = RAGRetriever(vector_store, embedding_manager)

In [ ]:
rag_retriver

In [ ]:
rag_retriver.retrieve("AI Engineer Roadmap ")